Будем учится интегрировать систему RAG для Альфреда итд, с использованием базы данных скаченных с HaginFace

In [ ]:
import datasets
from langchain_core.documents import Document

# Load the dataset
guest_dataset = datasets.load_dataset("agents-course/unit3-invitees", split="train")

# Convert dataset entries into Document objects
docs = [
    Document(
        page_content="\n".join([
            f"Name: {guest['name']}",
            f"Relation: {guest['relation']}",
            f"Description: {guest['description']}",
            f"Email: {guest['email']}"
        ]),
        metadata={"name": guest["name"]}
    )
    for guest in guest_dataset
]


In [ ]:
for i in docs:
  print(i)
  print('\n')

page_content='Name: Ada Lovelace
Relation: best friend
Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine.
Email: ada.lovelace@example.com' metadata={'name': 'Ada Lovelace'}


page_content='Name: Dr. Nikola Tesla
Relation: old friend from university days
Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about pigeons, so that might make for good small talk.
Email: nikola.tesla@gmail.com' metadata={'name': 'Dr. Nikola Tesla'}


page_content='Name: Marie Curie
Relation: no relation
Description: Marie Curie was a groundbreaking physicist and chemist, famous for her research on radioactivity.
Email: marie.curie@

In [ ]:
!pip install smolagents

In [ ]:
!pip install langchain_community

In [ ]:
!pip install rank_bm25

In [ ]:
from smolagents import Tool
from langchain_community.retrievers import BM25Retriever

class GuestInfoRetrieverTool(Tool):
    name = "guest_info_retriever"
    description = "Retrieves detailed information about gala guests based on their name or relation."
    inputs = {
        "query": {
            "type": "string",
            "description": "The name or relation of the guest you want information about."
        }
    }
    output_type = "string"

    def __init__(self, docs):
        self.is_initialized = False
        self.retriever = BM25Retriever.from_documents(docs)

    def forward(self, query: str):
        results = self.retriever.invoke(query)
        if results:
            return "\n\n".join([doc.page_content for doc in results[:3]])
        else:
            return "No matching guest information found."

# Initialize the tool
guest_info_tool = GuestInfoRetrieverTool(docs)

/tmp/ipykernel_5607/3722892848.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


In [ ]:
from smolagents import CodeAgent, InferenceClientModel

# Initialize the Hugging Face model
model = InferenceClientModel()

# Create Alfred, our gala agent, with the guest info tool
alfred = CodeAgent(tools=[guest_info_tool], model=model)

# Example query Alfred might receive during the gala
response = alfred.run("Tell me about our guest named 'Lady Ada Lovelace'.")

print("🎩 Alfred's Response:")
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Tell me about our guest named 'Lady Ada Lovelace'.                                                              │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  info = guest_info_retriever(query="Lady Ada Lovelace")                                                           
  final_answer(info)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Name: Ada Lovelace
Relation: best friend
Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for 
her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work
on Charles Babbage's Analytical Engine.
Email: ada.lovelace@example.com

Name: Marie Curie
Relation: no relation
Description: Marie Curie was a groundbreaking physicist and chemist, famous for her research on radioactivity.
Email: marie.curie@example.com

Name: Dr. Nikola Tesla
Relation: old friend from university days
Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless 
energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about 
pigeons, so that might make for good small talk.
Email: nikola.tesla@gmail.com

[Step 1: Duration 5.62 seconds| Input tokens: 2,087 | Output tokens: 539]

🎩 Alfred's Response:
Name: Ada Lovelace
Relation: best friend
Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine.
Email: ada.lovelace@example.com

Name: Marie Curie
Relation: no relation
Description: Marie Curie was a groundbreaking physicist and chemist, famous for her research on radioactivity.
Email: marie.curie@example.com

Name: Dr. Nikola Tesla
Relation: old friend from university days
Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about pigeons, so that might make for good small talk.
Email: nikola.tesla@gmail.com


новыый раздел тут мы Альфреду добовляем все инструменты которые сделали и новые тоже


In [ ]:
!pip install tools , retriever

ERROR: Invalid requirement: ',': Expected package name at the start of dependency specifier
    ,
    ^


In [ ]:
# Import necessary libraries
import random
from smolagents import CodeAgent, InferenceClientModel

# Import our custom tools from their modules
from tools import DuckDuckGoSearchTool, WeatherInfoTool, HubStatsTool
from retriever import load_guest_dataset

ModuleNotFoundError: No module named 'tools'

In [ ]:
import sys
sys.path.append('/content/Unit_3_Agentic_RAG/')

# Verify that the path has been added (optional)
print(sys.path)


['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/Unit_3_Agentic_RAG/']


In [ ]:
# Re-import the custom tools and dataset loader after adding their directory to sys.path
from tools import DuckDuckGoSearchTool, WeatherInfoTool, HubStatsTool
from retriever import load_guest_dataset

print("Custom tools and dataset loader imported successfully!")

Custom tools and dataset loader imported successfully!


In [ ]:
# Read the content of retriever.py
with open('/content/Unit_3_Agentic_RAG/retriever.py', 'r') as f:
    retriever_content = f.read()

# Replace the old import with the new one
updated_retriever_content = retriever_content.replace(
    'from langchain.docstore.document import Document',
    'from langchain_core.documents import Document'
)

# Write the updated content back to retriever.py
with open('/content/Unit_3_Agentic_RAG/retriever.py', 'w') as f:
    f.write(updated_retriever_content)

print("retriever.py updated successfully!")

retriever.py updated successfully!


In [ ]:
# Now, re-run the import cell after updating retriever.py
# Re-import the custom tools and dataset loader after adding their directory to sys.path
from tools import DuckDuckGoSearchTool, WeatherInfoTool, HubStatsTool
from retriever import load_guest_dataset

print("Custom tools and dataset loader imported successfully after retriever.py fix!")

Custom tools and dataset loader imported successfully after retriever.py fix!


In [ ]:
!pip install ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 71.0 MB/s eta 0:00:00


In [ ]:
# Initialize the Hugging Face model
model = InferenceClientModel()

# Initialize the web search tool
search_tool = DuckDuckGoSearchTool()

# Initialize the weather tool
weather_info_tool = WeatherInfoTool()

# Initialize the Hub stats tool
hub_stats_tool = HubStatsTool()

# Load the guest dataset
docs_for_retriever = load_guest_dataset()
# Initialize the guest info tool correctly
guest_info_retriever_tool = GuestInfoRetrieverTool(docs_for_retriever)

# Create Alfred with all the tools
alfred = CodeAgent(
    tools=[guest_info_retriever_tool, weather_info_tool, hub_stats_tool, search_tool],
    model=model,
    add_base_tools=True,  # Add any additional base tools
    planning_interval=3   # Enable planning every 3 steps
)

TypeError: 'GuestInfoRetrieverTool' object is not iterable

In [ ]:
# Test Alfred with a question that might use the WeatherInfoTool
response_weather = alfred.run("What's the weather like in London?")

print("🎩 Alfred's Response (Weather Query):")
print(response_weather)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What's the weather like in London?                                                                              │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
## 1. Facts survey
### 1.1. Facts given in the task
- Location: London

### 1.2. Facts to look up
- Current weather conditions for London (source: `weather_info` tool with location parameter "London")

### 1.3. Facts to derive
- None

## 2. Plan
1. Call the `weather_info` tool with the location parameter set to "London".
2. Use the result from the `weather_info` tool as the final answer.

```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  weather_data = weather_info(location="London")                                                                   
  final_answer(weather_data)                                                                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Weather in London: Rainy, 15°C

[Step 1: Duration 2.03 seconds| Input tokens: 2,424 | Output tokens: 264]

🎩 Alfred's Response (Weather Query):
Weather in London: Rainy, 15°C


In [ ]:
# Let's also test the guest info tool again, as Alfred now has multiple tools
response_guest_info = alfred.run("Tell me about Dr. Nikola Tesla.")

print("🎩 Alfred's Response (Guest Info Query):")
print(response_guest_info)
